In [1]:
import pandas as pd
from helpers import get_factor, get_price
pd.options.mode.chained_assignment = None

In [2]:
CDF = pd.read_csv("../production-v2/CDF.csv")
raw_sse = pd.read_csv("../basic/inspire_prtr_mapper.csv")
see = raw_sse.rename(columns={"InspireID_Betrieb": "plantid"})
seem = see[['plantid', 'sseid']]
#bpm['plantid'] = bpm['plantid'].apply(lambda x: str(x).replace('/', '_'))
seem['plantid'] = seem['plantid'].apply(lambda x: str(x).replace('/', '_'))

In [3]:
#CDF.sort_values(by=["produced_at", "variable"])

In [4]:
smard = pd.read_csv("Gro_handelspreise_202401010000_202501010000_Viertelstunde.csv", sep=";", na_values="-", decimal=",", thousands=".")
smardlog = smard[["Datum von", "Deutschland/Luxemburg [€/MWh] Originalauflösungen"]]
smardlog.rename(columns={"Datum von": "timestamp", "Deutschland/Luxemburg [€/MWh] Originalauflösungen": "price"}, inplace=True)
smardlog["timestamp"] = pd.to_datetime(smardlog["timestamp"], format="mixed")

In [5]:
smardlog.dtypes

timestamp    datetime64[ns]
price               float64
dtype: object

In [6]:
smardlog.to_excel("GHP_2023.xlsx")

In [7]:
smardlog.describe()

,timestamp,price
count,35136,35136.000000
mean,2024-07-02 00:26:55.573770496,78.512033
min,2024-01-01 00:00:00,-135.450000
25%,2024-04-01 12:56:15,55.560000
50%,2024-07-02 00:52:30,79.585000
75%,2024-10-01 12:48:45,101.340000
max,2024-12-31 23:45:00,936.280000
std,NaN,52.724158


In [8]:
smardlog.sort_values('price')

,timestamp,price
12721,2024-12-05 13:15:00,-135.45
12722,2024-12-05 13:30:00,-135.45
12723,2024-12-05 13:45:00,-135.45
12720,2024-12-05 13:00:00,-135.45
12725,2024-12-05 14:15:00,-132.85
...,...,...
29831,2024-06-11 17:45:00,820.11
33287,2024-12-12 17:45:00,936.28
33286,2024-12-12 17:30:00,936.28
33285,2024-12-12 17:15:00,936.28


In [9]:
co2s = pd.read_csv("../pollution/pollutants.csv")
nat_mp = pd.read_csv("nat_mapper_2025.csv")
plantlist = pd.read_csv("../basic/plants_2.csv")
nat_mp.fillna(0, inplace=True)

In [10]:
#co2s['year'] = co2s['year'] + 1

In [11]:
#co2s

In [12]:
CDF2 = CDF.loc[CDF.produced_at > "2023-12-31 23:50"].loc[CDF.produced_at < "2025-01-01 00:00"]

In [13]:
CDF2['produced_at'] = pd.to_datetime(CDF2['produced_at'])

In [14]:
CDF2

,produced_at,variable,value
78879,2024-01-01 00:00:00,SEE913896693631,0.0
78880,2024-01-01 01:00:00,SEE913896693631,0.0
78881,2024-01-01 02:00:00,SEE913896693631,0.0
78882,2024-01-01 03:00:00,SEE913896693631,0.0
78883,2024-01-01 04:00:00,SEE913896693631,0.0
...,...,...,...
43493831,2024-12-31 22:45:00,SEE960652233358,0.0
43493832,2024-12-31 23:00:00,SEE960652233358,0.0
43493833,2024-12-31 23:15:00,SEE960652233358,0.0
43493834,2024-12-31 23:30:00,SEE960652233358,0.0


In [15]:
CDF2a = CDF2.copy()#loc[~(CDF2.variable.str.contains("Unnamed"))]

In [16]:
#CDF2b = CDF2a.groupby('variable').resample('1h', on='produced_at').mean()

In [17]:
CDF3 = CDF2a

In [18]:
#CDF3 = CDF2.dropna()

In [19]:
len(CDF2) - len(CDF3)

0

In [20]:
len(CDF2.groupby('produced_at').sum())

35132

In [21]:
dataset = CDF3.merge(seem, left_on="variable", right_on="sseid", how="inner")

In [22]:
dataset2 = dataset.drop_duplicates(subset=['produced_at', 'variable'])

In [23]:
len(dataset)

3750341

In [24]:
dataset

,produced_at,variable,value,plantid,sseid
0,2024-01-01 00:00:00,SEE913896693631,0.0,06-02-B10117A007,SEE913896693631
1,2024-01-01 01:00:00,SEE913896693631,0.0,06-02-B10117A007,SEE913896693631
2,2024-01-01 02:00:00,SEE913896693631,0.0,06-02-B10117A007,SEE913896693631
3,2024-01-01 03:00:00,SEE913896693631,0.0,06-02-B10117A007,SEE913896693631
4,2024-01-01 04:00:00,SEE913896693631,0.0,06-02-B10117A007,SEE913896693631
...,...,...,...,...,...
3750336,2024-12-31 22:45:00,SEE960652233358,0.0,ST100125,SEE960652233358
3750337,2024-12-31 23:00:00,SEE960652233358,0.0,ST100125,SEE960652233358
3750338,2024-12-31 23:15:00,SEE960652233358,0.0,ST100125,SEE960652233358
3750339,2024-12-31 23:30:00,SEE960652233358,0.0,ST100125,SEE960652233358


In [25]:
magic = dataset2.groupby(["produced_at", "plantid"]).sum()

In [26]:
#magic

In [27]:
magic2 = magic[["value"]]

In [28]:
#magic2.sort_values(["produced_at", "value"])

In [29]:
production = magic2.reset_index()
production["produced_at"] = pd.to_datetime(production["produced_at"], format="mixed")

In [30]:
merged = production.merge(smardlog, left_on="produced_at", right_on="timestamp")
merged["revenue"] = merged["value"] * merged["price"]

In [31]:
merged_mytmp = merged[['produced_at', 'plantid', 'value', 'price', 'revenue']]

In [32]:
merged_mytmp.loc[merged_mytmp.plantid == "BB45025564"].sort_values('revenue').resample('1YE', on='produced_at').sum()

,plantid,value,price,revenue
produced_at,,,,
2024-12-31,BB45025564BB45025564BB45025564BB45025564BB4502...,9559700.0,2758598.8,8.005455e+08


In [33]:
#merged1a = merged.groupby('plantid').resample('1h', on='produced_at').mean().reset_index()

In [34]:
#merged1a.sort_values('revenue')

In [35]:
#merged

In [36]:
#production.sort_values(by=["plantid", "produced_at"])

In [37]:
#merged1a

In [38]:

#merged_tmp = merged.drop(columns=["timestamp"])
#merged1a = merged_tmp.set_index("produced_at", drop=True)


In [39]:
#merged1a = merged_tmp.set_index(['plantid']).sort_values(['plantid', 'produced_at'])

In [40]:
#merged2 = merged1a.resample("1h", on="produced_at").agg({'value':'sum', 'price':'sum', 'revenue': 'sum' })

In [41]:
#merged1a

In [42]:
#tmp1 = merged2.copy()
tmp1 = merged[["plantid", "value", "price", "revenue"]].groupby("plantid").sum()

In [43]:
tmp1.reset_index(inplace=True)

In [44]:
revenue = tmp1[["plantid", "revenue"]]

In [45]:
tmp1

,plantid,value,price,revenue
0,06-02-B10117A007,637346.0,689649.7,5.288343e+07
1,BB23020490,1376356.0,2758598.8,1.086602e+08
2,BB45025564,9559700.0,2758598.8,8.005455e+08
3,BB45025611,7007511.0,689649.7,6.055226e+08
4,BE166928,1171476.0,2758598.8,9.504822e+07
5,BE169709,362015.0,689649.7,3.070577e+07
6,BE172654,1297508.0,2758598.8,1.158482e+08
7,BE172656,962853.0,689649.7,8.590997e+07
8,BWpf-450-1020129-00000000,593836.0,2758598.8,6.110538e+07
9,BWpf-450-1195689-00000000,205400.0,2758598.8,1.626545e+07


In [46]:
#dataset.sort_values(["plantid", "produced_at"])

In [47]:
plantlist2 = plantlist[["plantid", "energysource"]]
plantlist3 = plantlist2.merge(nat_mp, on="plantid")

In [48]:
tmp0 = pd.merge(revenue, plantlist3, on="plantid")
tmp0["factor"] = tmp0["energysource"].apply(get_factor)
tmp0["fuel_price"] = tmp0["energysource"].apply(get_price)

In [49]:
#prod2 = prod.loc[prod.year == 2023].loc[prod.yearpower > 1000000]
co2s2 = co2s.loc[co2s.year == 2023].loc[co2s.pollutant == "CO2"]

In [50]:
co2s2.drop_duplicates(subset=["year", "plantid", "pollutant"], inplace=True)

In [51]:
co2s2

,year,plantid,pollutant,releases_to,amount,potency,unit_2,amount_2,pollutant2
56,2023,BB16018798,CO2,Air,2.850000e+08,9,Mio. t,0.285,CO2 [Mio. t]
92,2023,BB23020389,CO2,Air,4.340000e+08,9,Mio. t,0.434,CO2 [Mio. t]
181,2023,BB23020490,CO2,Air,3.015000e+09,9,Mio. t,3.015,CO2 [Mio. t]
388,2023,BB23022811,CO2,Air,1.500000e+08,9,Mio. t,0.150,CO2 [Mio. t]
443,2023,BB45025564,CO2,Air,1.413400e+10,9,Mio. t,14.134,CO2 [Mio. t]
...,...,...,...,...,...,...,...,...,...
16135,2023,ST18046,CO2,Air,2.210000e+08,9,Mio. t,0.221,CO2 [Mio. t]
16260,2023,TH30013152,CO2,Air,2.620000e+08,9,Mio. t,0.262,CO2 [Mio. t]
16293,2023,TH62013494,CO2,Air,1.290000e+08,9,Mio. t,0.129,CO2 [Mio. t]
16371,2023,TH72012874,CO2,Air,1.770000e+08,9,Mio. t,0.177,CO2 [Mio. t]


In [52]:
co2s3 = co2s2[["plantid", "amount_2"]]

In [53]:
tmp1 = pd.merge(tmp0, co2s3, on="plantid")

In [54]:
tmp2 = pd.merge(tmp1, production, on="plantid")

In [55]:
tmp2 = tmp1

In [56]:
coal_cost_per_t = 103.5# or 120
co2_cost = 65 # https://icapcarbonaction.com/system/files/ets_pdfs/icap-etsmap-factsheet-43.pdf
#electricity_price = 78.50

In [57]:
tmp2["co2_cost"] = (tmp2["amount_2"] * 10**6 - tmp2["free_co2s"]) * co2_cost / 10**6
tmp2["coal_cost"] = (tmp2["amount_2"] * 10**6 * 1/tmp2["factor"] * tmp2["fuel_price"]) / 10**6

In [58]:
tmp2["profit"] = (tmp2["revenue"] / 10**6) - (tmp2["co2_cost"] + tmp2["coal_cost"])

In [59]:
#tmp2

In [60]:
#tmp2.sort_values(by="free_co2s", ascending=False)

In [61]:
profit = tmp2[["plantid", "plantname", "revenue", "profit"]]
profit["revenue"] = profit["revenue"].apply(lambda x: x / 10**6)

In [62]:
profit.sort_values("profit", ascending=False)

,plantid,plantname,revenue,profit
16,BYS00048,Irsching 5 DT,359.523220,255.939887
24,NW100-0167182,SWD KWF GTKW,245.622271,163.397271
23,NI10257673950,Emsland B DT,282.762309,153.963495
47,SN70015796,Boxberg Block N,1069.532421,146.720561
40,NW900-9140178,Trianel Gaskraftwerk Hamm Block 10,137.017067,105.483734
29,NW300-0370387,DT Niehl 2 RheinEnergie,194.223969,100.448969
30,NW300-0877384,Weisweiler F,752.205333,94.452751
33,NW300-9046030,Knapsack I - Dampfturbine - DT 10,124.180906,89.622573
32,NW300-9002708,Dormagen DT,131.031108,66.681108
15,BYS00044,SWM HKW Sued GuD1 GT2,99.008555,63.533555


In [61]:
profit_combined = pd.concat([profit, profit2])

NameError: name 'profit2' is not defined

In [77]:
profit_final = profit_combined.drop_duplicates(subset="plantid", keep="first").sort_values('profit')

NameError: name 'profit_combined' is not defined

In [63]:
profit.to_csv("profit.csv", index=False)

In [189]:
pd.concat([profit2, profit, profit]).drop_duplicates(keep=False)

,plantid,plantname,revenue,profit
9,BYS00044,SWM HKW Sued GuD1 GT2,89.847656,52.437656
18,NW900-9141660,Trianel Kohlekraftwerk Lünen,290.568831,44.375619
21,SN80011277,Kraftwerk Lippendorf Block S,845.592635,286.401855


In [193]:
#profit_final.reset_index()

In [196]:
pd.concat([profit, profit2, profit2]).drop_duplicates(keep=False).sort_values('profit', ascending=False)

,plantid,plantname,revenue,profit
27,NW100-0248923,Neurath F,1581.931513,336.890044
24,NI10257673950,Emsland B DT,431.481929,295.721155
2,BB45025611,Kraftwerk Schwarze Pumpe Block A,877.033454,157.876547
30,NW300-0370387,DT Niehl 2 RheinEnergie,216.506863,117.616863
31,NW300-0877384,Weisweiler F,786.064712,81.693246
41,NW900-9140178,Trianel Gaskraftwerk Hamm Block 10,107.310336,74.057003
6,BE172656,GuD Marzahn Dampfturbine,120.058472,65.202539
15,BYS00044,SWM HKW Sued GuD1 GT2,88.946444,51.536444
38,NW900-0125939,Cuno Heizkraftwerk Herdecke BHKW1,69.697102,46.400435
34,NW300-9046030,Knapsack I - Dampfturbine - DT 10,69.154685,32.711351


In [116]:
#profit2 = profit

In [119]:
profit2.sort_values("profit", ascending=False)

,plantid,plantname,revenue,profit
21,SN80011277,Kraftwerk Lippendorf Block S,845.592635,286.401855
14,NW300-0326774,Niederaußem G,1245.726455,242.056864
10,BYS00048,Irsching 5 DT,341.278167,232.044833
20,SN70015796,Boxberg Block N,1202.545192,214.323757
13,NW100-0167182,SWD KWF GTKW,266.311410,179.601410
1,BB45025564,Kraftwerk Jänschwalde Block A,1180.306888,113.483473
3,BE172654,HKW Mitte Dampfturbine,142.743825,80.999075
9,BYS00044,SWM HKW Sued GuD1 GT2,89.847656,52.437656
18,NW900-9141660,Trianel Kohlekraftwerk Lünen,290.568831,44.375619
16,NW300-9002708,Dormagen DT,102.577362,34.671523
